# Auditoría de fuga y contrato de datos

Verificaciones de `src/audit.py` sobre la partición, los archivos, los metadatos y el pipeline. Cada una
tiene un criterio de aprobación fijado en `configs/base.yaml` (sección `audit`) o en
`configs/data_contract.yaml`, y un resultado «aprobado», «fallido» o «no disponible» si faltan los datos
que necesita. El resumen final reúne todas las verificaciones.

## 1. Configuración

`CALCULAR_SHA256` lee las 10015 imágenes para calcular sus SHA-256 si no están en
`data/interim/image_sha256.csv`.

In [ ]:
CALCULAR_SHA256 = False

In [ ]:
import pandas as pd

from src import audit, benchmark, data
from src.report import configure_display, show

configure_display()
cfg = data.load_config()
cfg_a = cfg["audit"]
definitivas = benchmark.load_runs_e2(cfg)
verificaciones = []
pd.Series({
    "umbral de pHash (Hamming)": cfg["phash_threshold"],
    "umbral de similitud coseno": cfg_a["cosine_threshold"],
    "proporción máxima con vecino": cfg_a["max_neighbor_frac"],
    "AUC máximo de un metadato": cfg_a["metadata_auc_max"],
    "AUC máximo adversarial": cfg_a["adversarial_auc_max"],
    "permutaciones": cfg_a["permutation"]["n_permutations"],
    "tolerancia de la permutación": cfg_a["permutation"]["tolerance"],
    "modelos con corridas": ", ".join(definitivas.models) or "ninguno",
}, name="valor").to_frame()

## 2. Partición

### 2.1 Intersección de lesiones

Lesiones (`lesion_id`) y grupos (`group_id`, que une además los duplicados confirmados) compartidos
entre train y validación de cada pliegue y entre el conjunto de entrenamiento y test.

In [ ]:
lesiones, v = audit.lesion_overlap(cfg)
verificaciones.append(v)
lesiones

### 2.2 Colisiones de SHA-256

Imágenes con el mismo SHA-256 y si alguna colisión cruza particiones (test o pliegues de train).

In [ ]:
hashes = audit.image_hashes(cfg, CALCULAR_SHA256)
colisiones = audit.sha256_collisions(cfg, hashes)
verificaciones.append(colisiones)
show(audit.detail(colisiones))

## 3. Vecinos casi duplicados

### 3.1 pHash

Para cada imagen de test, la distancia de Hamming mínima de su pHash a train; se marcan las que tienen un
vecino a distancia `phash_threshold` o menor.

In [ ]:
vecinos_phash = audit.phash_neighbors(cfg)
verificaciones.append(audit.neighbor_check(cfg, vecinos_phash, "phash"))
show(audit.detail(vecinos_phash))

Exactitud balanceada de test de cada modelo con todas las imágenes, sin las que tienen vecino y solo con ellas.

In [ ]:
sin_vecinos_phash = audit.performance_with_without(definitivas, cfg, audit.neighbor_flags(cfg, vecinos_phash, "phash"), "phash")
verificaciones.append(sin_vecinos_phash)
show(audit.detail(sin_vecinos_phash))

### 3.2 Embeddings

Similitud coseno máxima entre los embeddings de MobileNetV2 de cada imagen de test y los de train
(`audit.cosine_threshold`), y desempeño con y sin esos casos.

In [ ]:
vecinos_emb = audit.embedding_neighbors(cfg)
verificaciones.append(audit.neighbor_check(cfg, vecinos_emb, "embeddings"))
sin_vecinos_emb = audit.performance_with_without(definitivas, cfg, audit.neighbor_flags(cfg, vecinos_emb, "embeddings"), "embeddings")
verificaciones.append(sin_vecinos_emb)
show({"Vecinos": vecinos_emb, "Desempeño con y sin vecinos": audit.detail(sin_vecinos_emb)})

## 4. Metadatos

### 4.1 AUC univariado

AUC uno contra el resto de cada metadato para cada clase en test, con la relación categoría-clase
estimada en train; el modelo no usa metadatos, así que un AUC alto señala un posible atajo si la imagen
codifica el metadato.

In [ ]:
metadatos = data.load_metadata(cfg)
auc_metadatos = audit.metadata_auc(cfg, metadatos)
verificaciones.append(auc_metadatos)
show(audit.detail(auc_metadatos))

### 4.2 Validación adversarial

AUC de una regresión logística que distingue train de test con los embeddings (la misma función que el
EDA, `src.eda.adversarial_validation`).

In [ ]:
adversarial = audit.adversarial_check(cfg)
verificaciones.append(adversarial)
show(audit.detail(adversarial))

## 5. Sanidad del pipeline

### 5.1 Etiquetas permutadas

Regresión logística sobre los embeddings entrenada con las etiquetas de train permutadas: su exactitud
balanceada debe quedar cerca del azar, 1 / K.

In [ ]:
permutacion = audit.permutation_check(cfg)
verificaciones.append(permutacion)
show(audit.detail(permutacion))

### 5.2 Ajustes solo con train

Pruebas con datos sintéticos: al cambiar los datos de validación o test no cambia nada de lo que se
ajusta con train (partición, pesos de clase, escalado y regresión logística, prior del baseline trivial)
ni lo que se ajusta con validación (temperatura y umbral conforme).

In [ ]:
sinteticas = audit.synthetic_fit_checks(cfg)
verificaciones.append(sinteticas)
show(audit.detail(sinteticas))

## 6. Contrato de datos

### 6.1 Esquema

Validación de `HAM10000_metadata.csv` contra `configs/data_contract.yaml`: tipos, faltantes, unicidad,
patrones, valores permitidos, rangos y consistencia dentro de cada lesión. Los metadatos se validan
también al cargarlos en la evaluación (`src.data.load_metadata_validated`).

In [ ]:
contrato = audit.contract_check(cfg, metadatos)
verificaciones.append(contrato)
show(audit.detail(contrato))

### 6.2 Diccionario de atributos

`data/diccionario.csv`, generado desde el contrato y los metadatos.

In [ ]:
diccionario = audit.build_dictionary(cfg, metadatos)
diccionario

### 6.3 Observaciones por población

Imágenes, lesiones y clases en cada población de la partición.

In [ ]:
poblaciones = audit.population_counts(cfg)
verificaciones.append(poblaciones)
show(audit.detail(poblaciones))

## 7. Resumen

Todas las verificaciones con su criterio, valor y resultado.

In [ ]:
audit.collect(verificaciones)